In [5]:
### Chapter 1: Common Conventions and API Elements of scikit-learn
Notebook ini berisi implementasi lengkap bab pertama mengenai filosofi desain, penggunaan estimator, transformer, predictor, pipeline, serta manajemen parameter di scikit-learn.

SyntaxError: invalid syntax (2731105216.py, line 2)

# Chapter 1: Common Conventions and API Elements of scikit-learn

## 1. Chapter Summary
Bab ini membahas prinsip desain arsitektur dan konvensi antarmuka (*API conventions*) yang mendasari pustaka **scikit-learn**. Konsep utama yang dipelajari mencakup:
- **Design Philosophy**: Prinsip konsistensi, kesederhanaan, modularitas, dan reusabilitas.
- **Estimator**: Objek pemodelan yang mempelajari representasi atau parameter data melalui metode `.fit()`.
- **Transformer**: Objek rekayasa data yang memproses fitur melalui metode `.transform()` atau `.fit_transform()`.
- **Predictor**: Model yang menghasilkan inferensi data uji melalui `.predict()`.
- **Pipelines**: Penggabungan alur transformasi dan estimasi secara sekuensial guna mencegah *data leakage*.
- **Parameter & Metadata Management**: Pengaturan hyperparameter terprogram (`get_params()`, `set_params()`) serta metadata tags.

In [6]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sklearn
print(f"Versi scikit-learn : {sklearn.__version__}")
print(f"Versi NumPy        : {np.__version__}")

Versi scikit-learn : 1.6.1
Versi NumPy        : 2.1.3


## 2. Theoretical Deep-Dive: Core API Elements & Philosophy

### 2.1 Desain Antarmuka Terpadu (Unified API)
scikit-learn dibangun di atas antarmuka berbasis *Object-Oriented Programming* (OOP) yang seragam di seluruh algoritma:
1. **Consistency**: Seluruh modul mematuhi nama fungsi yang identik:
   - `fit(X, y)`: Melatih model atau mempelajari statistik fitur.
   - `predict(X)`: Melakukan inferensi target pada data baru.
   - `transform(X)`: Mengubah data menggunakan statistik yang telah dipelajari.
2. **Sensible Defaults**: Setiap model telah diinisialisasi dengan hyperparameter standar yang optimal untuk eksperimen awal.
3. **Modularity**: Komponen pembersihan data, penskalaan, dan algoritma pembelajaran dapat dipertukarkan dengan mudah.

### 2.2 Estimator Supervised vs Unsupervised
- **Supervised Learning**: Model mempelajari relasi $y = f(X)$. Pada regresi linear sederhana, model meminimalkan fungsi *Ordinary Least Squares* (OLS):
  $$\min_{w, b} \sum_{i=1}^{n} (y_i - (w \cdot x_i + b))^2$$
  Solusi analitiknya dipelajari pada tahap `.fit(X, y)` dan prediksi dilakukan melalui `.predict(X)`.
- **Unsupervised Learning**: Data tidak memiliki label target $y$. Contohnya K-Means clustering yang mengoptimalkan penempatan sentroid:
  Metode praktis `.fit_predict(X)` digunakan untuk mempelajari sekaligus mempartisi data latih ke dalam klaster dalam satu langkah.

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

# A. Implementasi Estimator Terbimbing (Supervised): Linear Regression
X_train = np.array([[1], [2], [3], [4], [5]])
y_train = np.array([1, 2, 3, 3.5, 5])

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)

X_test = np.array([[6], [7]])
predictions = reg_model.predict(X_test)

print("=== Hasil Regresi Linear ===")
print("Prediksi untuk input [6, 7] :", predictions)
print("Bobot kemiringan (coef_)    :", reg_model.coef_)
print("Titik potong (intercept_)   :", reg_model.intercept_)
print(f"R-squared Score             : {reg_model.score(X_train, y_train):.4f}")

# B. Implementasi Estimator Tak Terbimbing (Unsupervised): K-Means
kmeans_model = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans_model.fit_predict(X_train)

print("\n=== Hasil K-Means Clustering ===")
print("Label Klaster   :", cluster_labels)
print("Titik Sentroid  :", kmeans_model.cluster_centers_.flatten())

=== Hasil Regresi Linear ===
Prediksi untuk input [6, 7] : [5.75 6.7 ]
Bobot kemiringan (coef_)    : [0.95]
Titik potong (intercept_)   : 0.04999999999999938
R-squared Score             : 0.9810

=== Hasil K-Means Clustering ===
Label Klaster   : [0 0 0 1 1]
Titik Sentroid  : [2.  4.5]


## 3. Data Transformations & Pipelines

### 3.1 Penskalaan Fitur (Z-Score Standardization)
Transformer seperti `StandardScaler` mengubah distribusi fitur sehingga memiliki nilai mean nol ($\mu = 0$) dan variansi satu ($\sigma = 1$):
$$z = \frac{x - \mu}{\sigma}$$
- **Aturan Pemisahan Data**: `.fit_transform()` hanya dipanggil pada **data latih (training set)** untuk menghitung $\mu$ dan $\sigma$. Pada **data uji (test set)**, hanya dipanggil `.transform()` menggunakan parameter yang telah disimpan agar tidak terjadi kebocoran informasi (*data leakage*).

### 3.2 Alur Otomasi Menggunakan Pipeline
Kelas `Pipeline` merangkai transformasi dan estimator secara sekuensial. Alur eksekusinya:
- `fit()`: Menjalankan `fit_transform()` pada setiap transformer, kemudian memanggil `fit()` pada estimator akhir.
- `predict()`: Menjalankan `transform()` pada seluruh transformer secara berurutan, lalu memanggil `predict()` pada model akhir.

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 1. Transformasi Data dengan StandardScaler
X_sample = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

print("=== Pengujian Transformer Tunggal ===")
print("Data Asli:\n", X_sample)
print("Data Hasil Standarisasi:\n", X_scaled)
print("Rata-rata fitur (scaler.mean_) :", scaler.mean_)
print("Standar deviasi (scaler.scale_):", scaler.scale_)

# 2. Rangkaian End-to-End dengan Pipeline
X_cls, y_cls = make_classification(
    n_samples=200, n_features=5, n_informative=3, n_redundant=1, random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_tr, y_tr)

train_acc = pipeline.score(X_tr, y_tr)
test_acc = pipeline.score(X_te, y_te)

print("\n=== Evaluasi Pipeline ===")
print("Tahapan Pipeline :", [step[0] for step in pipeline.steps])
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy : {test_acc:.4f}")

=== Pengujian Transformer Tunggal ===
Data Asli:
 [[1. 2.]
 [3. 4.]
 [5. 6.]]
Data Hasil Standarisasi:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
Rata-rata fitur (scaler.mean_) : [3. 4.]
Standar deviasi (scaler.scale_): [1.63299316 1.63299316]

=== Evaluasi Pipeline ===
Tahapan Pipeline : ['scaler', 'classifier']
Training Accuracy: 0.9267
Testing Accuracy : 0.8800


## 4. Hyperparameter Management & Estimator Metadata

### 4.1 Akses dan Modifikasi Parameter
Parameter bawaan model dapat diinspeksi menggunakan metode `.get_params()` dan dimodifikasi secara terprogram menggunakan `.set_params()`. Pola ini memudahkan integrasi dengan fungsi optimasi seperti `GridSearchCV`.

### 4.2 Metadata dan Estimator Tags
Setiap estimator memuat metadata/tags internal untuk memvalidasi kompatibilitas fitur, misalnya dukungan klasifikasi multikelas atau toleransi terhadap masukan bernilai kosong (*missing values*).

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

# Menetapkan hyperparameter
rf.set_params(n_estimators=100, max_depth=8, criterion='gini', random_state=42)

# Mengambil parameter model
params = rf.get_params()
print("=== Parameter Terkonfigurasi pada Random Forest ===")
print("n_estimators :", params['n_estimators'])
print("max_depth    :", params['max_depth'])
print("criterion    :", params['criterion'])
print("random_state :", params['random_state'])

# Memeriksa kemampuan model melalui tags bawaan
tags = rf.__sklearn_tags__()
print("\n=== Metadata / Estimator Tags ===")
print("Estimator Type   :", type(rf).__name__)
print("Array API Support:", tags.array_api_support)

=== Parameter Terkonfigurasi pada Random Forest ===
n_estimators : 100
max_depth    : 8
criterion    : gini
random_state : 42

=== Metadata / Estimator Tags ===
Estimator Type   : RandomForestClassifier
Array API Support: False


## 5. Analysis & Conclusions
1. **Verifikasi Estimator Linear**: Model regresi linear menghasilkan persamaan garis $\hat{y} = 0.95x + 0.05$ dengan nilai evaluasi $R^2 = 0.9810$. Hal ini menunjukkan bahwa sebagian besar variansi target dapat diprediksi secara akurat oleh fitur independen.
2. **Konsistensi Pipeline**: Integrasi `StandardScaler` dan `LogisticRegression` ke dalam objek `Pipeline` menghasilkan performa uji (*testing accuracy*) sebesar 0.8800. Pipeline menjamin pemrosesan data uji menggunakan parameter statistik dari data latih secara konsisten tanpa risiko kebocoran data (*data leakage*).
3. **Pengelolaan Fleksibel**: Melalui kombinasi `get_params()` dan `set_params()`, konfigurasi model dapat diekstrak atau diubah tanpa perlu menginstansiasi ulang kelas model dari awal.